# Natural Language Processing INM343
### Comparative Legal Clause Classification



#### The notebook compares four families of approaches:

- dummy baselines for lower-bound context
- classical sparse-text models using TF-IDF features
- an optional fine-tuned transformer classifier
- an optional Qwen2.5-Instruct prompting baseline



## 1. Colab Setup, Imports, and Configuration

This stage prepares the runtime so the same notebook can run locally or in Google Colab. In Colab, upload, unzip, sync, or clone the whole project folder, not just this notebook.



Importing Libraries and Modules

In [ ]:
from pathlib import Path

import importlib.util
import os
import subprocess
import sys
import numpy as np
import pandas as pd

import json
import math
import re

import pandas as pd
from datetime import datetime, timezone
from IPython.display import display

File Setup

In [ ]:
# @title
PROJECT_ROOT_OVERRIDE = os.environ.get("LEDGAR_PROJECT_ROOT", "").strip()
AUTO_MOUNT_GOOGLE_DRIVE = True
INSTALL_REQUIREMENTS_IN_COLAB = True


def running_in_colab() -> bool:
    return "COLAB_RELEASE_TAG" in os.environ or importlib.util.find_spec("google.colab") is not None


IN_COLAB = running_in_colab()

In [ ]:
# @title

if IN_COLAB:
    print("Google Colab runtime detected.")

if IN_COLAB and AUTO_MOUNT_GOOGLE_DRIVE:
    try:
        if Path("/content/drive/MyDrive").exists():
            print("Google Drive is already available.")
        else:
            from google.colab import drive

            drive.mount("/content/drive")
    except Exception as exc:
        print(f"Google Drive mount skipped/failed: {type(exc).__name__}: {exc}")


def looks_like_project_root(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "modules").is_dir()


def project_root_candidates_near(path: Path) -> list[Path]:
    path = path.expanduser()
    candidates = [path, *path.parents]
    if path.exists() and path.is_dir():
        for pattern in (
            "pyproject.toml",
            "*/pyproject.toml",
            "*/*/pyproject.toml",
            "*/*/*/pyproject.toml",
        ):
            candidates.extend(pyproject.parent for pyproject in path.glob(pattern))
    deduped = []
    seen = set()
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except Exception:
            resolved = candidate
        if resolved not in seen:
            deduped.append(resolved)
            seen.add(resolved)
    return deduped


def parent_search(start: Path) -> Path | None:
    for candidate in project_root_candidates_near(start):
        if looks_like_project_root(candidate):
            return candidate
    return None


def common_colab_candidates() -> list[Path]:
    candidates = [
        Path("/content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing"),
    ]
    for base in (Path("/content"), Path("/content/drive/MyDrive")):
        if base.exists():
            for pattern in (
                "Natural-Language-Processing",
                "*/Natural-Language-Processing",
                "*/*/Natural-Language-Processing",
                "*/*/*/Natural-Language-Processing",
            ):
                candidates.extend(base.glob(pattern))
    return candidates


def find_notebook_project_root() -> Path:
    if PROJECT_ROOT_OVERRIDE:
        override = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        for candidate in project_root_candidates_near(override):
            if looks_like_project_root(candidate):
                if candidate != override:
                    print(f"PROJECT_ROOT_OVERRIDE pointed to a parent folder; using nested project root: {candidate}")
                return candidate
        raise FileNotFoundError(
            f"PROJECT_ROOT_OVERRIDE does not contain pyproject.toml and modules/, and no nested project root was found under it: {override}\n"
            "Check the Drive folder path, or run this diagnostic: list(Path('/content/drive/MyDrive').glob('**/pyproject.toml'))"
        )

    root = parent_search(Path.cwd())
    if root is not None:
        return root

    if IN_COLAB:
        for candidate in common_colab_candidates():
            if candidate.exists() and looks_like_project_root(candidate):
                return candidate.resolve()

    raise FileNotFoundError(
        "Could not find the project root containing pyproject.toml and modules/.\n"
        "In Colab, upload or clone the whole repository, then set PROJECT_ROOT_OVERRIDE "
        "near the top of this cell to that folder. Current working directory: "
        f"{Path.cwd()}"
    )

In [ ]:
# Find the project root and add it to sys.path so that imports work, even if the notebook is opened in a subfolder or outside the project.
PROJECT_ROOT = find_notebook_project_root()
os.environ["LEDGAR_PROJECT_ROOT"] = str(PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# List of (import_name, pip_name) for packages commonly used in notebooks. pip_name can be None if it's the same as import_name.
REQUIRED_NOTEBOOK_PACKAGES = [
    ("pandas", "pandas"),
    ("datasets", "datasets"),
    ("huggingface_hub", "huggingface_hub"),
    ("sklearn", "scikit-learn"),
    ("joblib", "joblib"),
    ("matplotlib", "matplotlib"),
]

# In Colab, install all requirements from requirements-colab.txt if any are missing, to avoid multiple pip installs.
def ensure_notebook_package(import_name: str, pip_name: str | None = None) -> None:
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])

# Check for missing imports before installing requirements in Colab, to avoid unnecessary pip installs and speed up notebook startup.
missing_imports = [name for name, _ in REQUIRED_NOTEBOOK_PACKAGES if importlib.util.find_spec(name) is None]
requirements_path = PROJECT_ROOT / "requirements-colab.txt"


if IN_COLAB and INSTALL_REQUIREMENTS_IN_COLAB and requirements_path.exists() and missing_imports:
    print(f"Installing Colab requirements from {requirements_path}.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)])
else:
    for import_name, pip_name in REQUIRED_NOTEBOOK_PACKAGES:
        ensure_notebook_package(import_name, pip_name)

Custom Modules and Libraries

In [ ]:
from modules.data_setup import (
    adapt_cuad_to_clause_classification,
    build_project_paths,
    download_cuad_if_missing,
    load_cuad_raw_files,
    load_or_download_ledgar,
    normalise_whitespace,
    print_dataset_availability,
    seed_everything,
)
from modules.preprocessing import (
    bpe_encode_text,
    bpe_encode_word,
    clean_html_entities,
    corpus_word_frequencies,
    create_ledgar_eda,
    legal_safe_tokenise,
    negation_aware_tokenise,
    preprocess_ledgar,
    preprocessing_technique_rundown,
    regex_tokenise,
    train_bpe_tokeniser,
    write_preprocessing_rundown,
)
from modules.baselines import run_baseline_experiments
from modules.classical_models import run_classical_experiments
from modules.sequence_model import SequenceModelConfig, train_sequence_classifier
from modules.transformer_model import train_transformer_classifier
from modules.transformer_hpt import TransformerHPTConfig, run_two_stage_transformer_hpt
from modules.qwen_prompting import run_qwen_baseline
from modules.agentic_review import run_agentic_review
from modules.evaluation import save_final_comparison
from modules.error_analysis import run_error_analysis
from modules.wandb_reporting import finish_wandb_run, log_wandb_outputs, start_wandb_run



| Setting | Value | Purpose |
|---|---:|---|
| `SEED` | `42` | Makes sampling, baseline randomness, and train/test helper behavior reproducible. |
| `DATASET_NAME` | `LEDGAR` | Keeps the main experiment scoped to LEDGAR clause classification. |
| `TOP_K_LABELS` | `20` | Restricts the task to the 20 most frequent training labels for a manageable coursework experiment. |
| `RUN_CLASSICAL_MODELS` | `True` | Enables TF-IDF model experiments. |
| `RUN_TRANSFORMER` | `True` | Attempts transformer fine-tuning only when the runtime can support it. |
| `RUN_QWEN_BASELINE` | `True` | Attempts Qwen prompting only when GPU/model loading is available. |
| `RUN_AGENTIC_EXTENSION` | `True` | Enables a small review workflow demonstration, not an autonomous agent. |
| `RUN_WANDB` | `True` | Sends metrics and safe artifacts to W&B when credentials are available. |

Model and feature hyperparameters declared here:

| Component | Hyperparameters |
|---|---|
| TF-IDF search | `max_features` in `[10000, 30000]`; `ngram_range` in `[(1, 1), (1, 2)]`; `lowercase=True`; `stop_words=None` |
| Transformer | `distilbert-base-uncased`; `max_length=256` |
| Optional legal transformer | `nlpaueb/legal-bert-base-uncased` can be substituted manually if GPU resources allow |
| Qwen prompting | `Qwen/Qwen2.5-3B-Instruct`; test sample size `200`; one few-shot example per class when available |
| W&B logging | Uses `WANDB_API_KEY` from Colab Secrets or the environment; text-containing prediction/error tables are not uploaded unless `WANDB_LOG_TEXT_TABLES=True` |

Explainability note: keeping all configuration values in one cell makes it clear which choices affect runtime cost, model capacity, and evaluation scope.

In [ ]:
SEED = 42

DATASET_NAME = "LEDGAR"

TOP_K_LABELS = 20

MAX_FEATURES_LIST = [10000, 30000]

NGRAM_RANGES = [(1, 1), (1, 2)]

RUN_CLASSICAL_MODELS = True

RUN_TRANSFORMER = True

RUN_TRANSFORMER_HPT = False

HPT_RANDOM_TRIALS = 8

HPT_BAYES_TRIALS = 8

RUN_QWEN_BASELINE = True

RUN_AGENTIC_EXTENSION = True

RUN_NAIVE_BAYES = True

RUN_SEQUENCE_MODEL = False

RUN_WANDB = True

WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "ledgar-clause-classification")

WANDB_ENTITY = os.environ.get("WANDB_ENTITY", "").strip() or None

WANDB_MODE = os.environ.get("WANDB_MODE", "online")

WANDB_LOG_ARTIFACTS = False

WANDB_LOG_TEXT_TABLES = False

WANDB_LOG_MODEL_FILES = False

TRANSFORMER_MODEL_NAME = "distilbert-base-uncased"

OPTIONAL_LEGAL_MODEL_NAME = "nlpaueb/legal-bert-base-uncased"

QWEN_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

MAX_TRANSFORMER_LENGTH = 256

QWEN_EVAL_SAMPLE_SIZE = 200

QWEN_FEW_SHOT_EXAMPLES_PER_CLASS = 1


DOWNLOAD_LEDGAR_IF_MISSING = True

DOWNLOAD_CUAD_IF_MISSING = True

USE_HF_CACHE = True

FORCE_REDOWNLOAD = False

paths = build_project_paths(PROJECT_ROOT)
DEVICE = seed_everything(SEED)


Weights and Biases Setup

In [ ]:
print(f"Project root: {paths.project_root}")
print(f"Colab runtime: {IN_COLAB}")
print(f"Raw LEDGAR directory: {paths.ledgar_raw_dir}")
print(f"Results directory: {paths.results_dir}")
print(f"Device: {DEVICE}")

In [ ]:
try:
    import torch

    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU count: {torch.cuda.device_count()}")
        for gpu_idx in range(torch.cuda.device_count()):
            print(f"GPU {gpu_idx}: {torch.cuda.get_device_name(gpu_idx)}")
    else:
        print("GPU is unavailable. Transformer/Qwen sections will skip or reduce work gracefully.")
except Exception:
    print("PyTorch is unavailable. Transformer/Qwen sections will skip if they require it.")

# Robust W&B handling for Colab stage-by-stage runs.
# The notebook no longer keeps one long W&B run open across preprocessing, classical models, exports, and audits.
import os

os.environ.setdefault("WANDB_CONSOLE", "off")
os.environ.setdefault("WANDB_DISABLE_CODE", "true")
os.environ.setdefault("WANDB_START_METHOD", "thread")

WANDB_ACTIVE = False
wandb_run = None


def maybe_start_wandb_run(stage_name: str, config: dict | None = None):
    """Start a fresh W&B run for one major training/evaluation stage.

    This avoids false W&B "crashed" states caused by leaving a single run open
    across many independent notebook stages in Colab.
    """
    global WANDB_ACTIVE, wandb_run

    WANDB_ACTIVE = False
    wandb_run = None

    if not globals().get("RUN_WANDB", False):
        print(f"W&B disabled for stage: {stage_name}")
        return None

    try:
        import wandb

        # Close any previously open run before starting a new stage run.
        if wandb.run is not None:
            print("Existing W&B run found; finishing it before starting a new one.")
            try:
                wandb.finish(exit_code=0)
            except TypeError:
                wandb.finish()

        try:
            settings = wandb.Settings(start_method="thread")
        except TypeError:
            settings = wandb.Settings()

        run = wandb.init(
            project=globals().get("WANDB_PROJECT", "ledgar-clause-classification"),
            entity=globals().get("WANDB_ENTITY", None),
            mode=globals().get("WANDB_MODE", "online"),
            name=stage_name,
            config=config or {},
            reinit=True,
            settings=settings,
        )

        WANDB_ACTIVE = True
        wandb_run = run
        print(f"W&B active for stage: {stage_name}")
        return run

    except Exception as exc:
        WANDB_ACTIVE = False
        wandb_run = None
        print(f"W&B could not start for {stage_name}: {type(exc).__name__}: {exc}")
        print("Continuing without W&B.")
        return None


def finish_wandb_run(*args, **kwargs):
    """Finish the current W&B run cleanly and release memory.

    Accepts *args/**kwargs for compatibility with older cells that called
    finish_wandb_run(wandb_run).
    """
    global WANDB_ACTIVE, wandb_run

    try:
        import wandb
        if wandb.run is not None:
            try:
                wandb.finish(exit_code=0)
            except TypeError:
                wandb.finish()
            print("W&B run finished cleanly.")
    except Exception as exc:
        print(f"W&B finish warning: {type(exc).__name__}: {exc}")

    try:
        import gc
        gc.collect()
    except Exception:
        pass

    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass

    WANDB_ACTIVE = False
    wandb_run = None


## 2. Dataset Download and Raw Setup

This stage obtains the raw datasets without training or preprocessing models. LEDGAR remains the main classification dataset. CUAD is treated separately because it is structured as a contract-review question-answering/span-extraction dataset rather than a direct clause-classification dataset.

Data governance choices:

- LEDGAR is loaded from Hugging Face with `load_dataset("coastalcph/lex_glue", "ledgar")` when local JSONL files are missing.
- Official LEDGAR train, validation, and test splits are preserved when available.
- Raw LEDGAR split exports are saved under `data/raw/lexglue_ledgar/` as JSONL files.
- CUAD raw files are downloaded from `theatticusproject/cuad` when available, but CUAD is not merged with LEDGAR.
- If CUAD is missing, the notebook prints a clear message and continues with LEDGAR.

Inputs and outputs:

| Input | Output |
|---|---|
| Hugging Face LEDGAR or local JSONL | `ledgar_raw_splits` dictionary with train/validation/test DataFrames |
| Optional CUAD raw files | `cuad_clause_df` containing extracted span-level examples for optional inspection |

This stage deliberately does not select labels, encode classes, train models, or compute metrics.


CAUD Dataset

In [ ]:
ledgar_raw_splits = load_or_download_ledgar(
    paths,
    download_if_missing=DOWNLOAD_LEDGAR_IF_MISSING,
    force_redownload=FORCE_REDOWNLOAD,
)

cuad_json_path, master_clauses_path = download_cuad_if_missing(
    paths,
    download_if_missing=DOWNLOAD_CUAD_IF_MISSING,
    force_redownload=FORCE_REDOWNLOAD,
)

raw_cuad_json, master_clauses_df = load_cuad_raw_files(cuad_json_path, master_clauses_path)
cuad_clause_df = adapt_cuad_to_clause_classification(raw_cuad_json)
print_dataset_availability(ledgar_raw_splits, cuad_json_path, master_clauses_path, cuad_clause_df)

## 3. LEDGAR Preprocessing and EDA

This stage is intentionally split into small cells so each lecture/lab technique can be inspected before the full dataset is processed.

Preprocessing changes the raw text into a cleaner form. Feature extraction converts text into numeric vectors. Modelling starts only after those vectors are produced.


### Cell 1 - Separate preprocessing, feature extraction, and modelling

| Stage | What happens here | Examples in this notebook |
|---|---|---|
| Preprocessing | Clean or tokenise raw clause text | HTML/entity cleanup, whitespace normalisation, regex tokens, negation-aware tokens, BPE inspection |
| Feature extraction | Convert text/tokens into numeric vectors | BoW, TF-IDF, unigrams, bigrams |
| Modelling | Fit a classifier using features and labels | Logistic Regression, Linear SVM, Naive Bayes |

Logistic Regression is therefore not preprocessing; it appears later as a model.


In [ ]:
from modules.preprocessing import (
    bpe_encode_text,
    bpe_encode_word,
    clean_html_entities,
    corpus_word_frequencies,
    create_ledgar_eda,
    legal_safe_tokenise,
    negation_aware_tokenise,
    preprocess_ledgar,
    preprocessing_technique_rundown,
    regex_tokenise,
    train_bpe_tokeniser,
    write_preprocessing_rundown,
)


In [ ]:
#  Show one raw contract clause example before preprocessing.
if ledgar_raw_splits:
    raw_train_df = ledgar_raw_splits["train"]
    raw_text_column = next(column for column in ("text", "provision", "clause", "contract_text") if column in raw_train_df.columns)
    raw_clause = str(raw_train_df.iloc[0][raw_text_column])
else:
    raw_text_column = "text"
    raw_clause = "The Borrower shall not be liable for any indirect damages &amp; shall give notice under Section 5.1."

print(raw_clause[:1000])


In [ ]:
# Apply whitespace normalisation.
whitespace_clean_clause = normalise_whitespace(html_clean_clause)
print(whitespace_clean_clause[:1000])


In [ ]:
# Run regex tokenisation.
regex_tokens = regex_tokenise(whitespace_clean_clause)
print(regex_tokens[:80])
print(f"Token count: {len(regex_tokens)}")


In [ ]:
# Run legal-safe lowercased tokenisation.
# This lowercases tokens for feature extraction without overwriting the stored clause text.
legal_tokens = legal_safe_tokenise(whitespace_clean_clause)
print(legal_tokens[:80])


In [ ]:
# Run Week 2 negation-aware tokenisation.
negation_tokens = negation_aware_tokenise(whitespace_clean_clause)
print(negation_tokens[:100])


In [ ]:
# Show why stopword removal is skipped for legal clauses.
# These words are often treated as stopwords in generic NLP, but they can change legal meaning.
legal_stopword_examples = {"no", "not", "shall", "may", "unless", "except", "without"}
kept_legal_tokens = [token for token in legal_tokens if token in legal_stopword_examples]

print("Legal stopword-like tokens kept:", kept_legal_tokens)
print("Default decision: do not remove stopwords for contract clause classification.")


In [ ]:
# Train a small Week 2/3 BPE tokenizer on training clause samples.
if ledgar_raw_splits:
    bpe_training_texts = ledgar_raw_splits["train"][raw_text_column].astype(str).head(250).tolist()
else:
    bpe_training_texts = [whitespace_clean_clause]

bpe_word_counts = corpus_word_frequencies(bpe_training_texts, max_words=2000)
bpe_merges, bpe_vocab = train_bpe_tokeniser(bpe_word_counts, num_merges=50)

print(f"BPE training words: {len(bpe_word_counts)}")
print(f"BPE merges learned: {len(bpe_merges)}")
print(list(bpe_merges.items())[:10])


In [ ]:
# Run BPE encoding/OOV examples.
for word in ["lowest", "lover", "newly", "unwanted", "indemnification", "xyz"]:
    print(f"{word:20} -> {bpe_encode_word(word, bpe_merges)}")

print("Clause BPE preview:")
print(bpe_encode_text(whitespace_clean_clause, bpe_merges)[:100])


In [ ]:
# Preprocess full LEDGAR splits and save processed outputs.
processed_splits, label2id, id2label = preprocess_ledgar(
    ledgar_raw_splits,
    paths,
    top_k_labels=TOP_K_LABELS,
    dataset_name=DATASET_NAME,
)

split_summary = create_ledgar_eda(processed_splits, paths.results_dir)

if processed_splits:
    train_df = processed_splits["train"]
    validation_df = processed_splits["validation"]
    test_df = processed_splits["test"]
    label_names = [id2label[i] for i in sorted(id2label)]
    display(split_summary)
    display(pd.DataFrame({"label": label_names}))
else:
    train_df = validation_df = test_df = pd.DataFrame(
        columns=["text", "label", "label_id", "split", "source_dataset"]
    )
    label_names = []
    print("Main LEDGAR experiment cannot run without LEDGAR data.")


In [ ]:
# Write a readable rundown of preprocessing and feature techniques.
rundown_path = write_preprocessing_rundown(paths.project_root / "outputs" / "preprocessing_techniques.md")
print(f"Wrote: {rundown_path}")
print(preprocessing_technique_rundown())


## 4. Shared Result State

This short stage creates shared containers used by the later model sections.

- `completed_results` stores one row per completed or skipped model run.
- `prediction_tables` stores per-example predictions for error analysis.
- `trained_models` stores reusable fitted model objects when available.

Governance purpose: every model section appends to the same result structure, so the final comparison table is generated from actual run outputs rather than manually entered values.


Variable Initialization

In [ ]:
completed_results = []

prediction_tables = {}

trained_models = {}

## 5. Dummy Baselines

This stage evaluates non-learning baselines. These baselines are important because they establish a minimum reference point before interpreting more complex models.

Baselines used:

| Model | Behavior | Why it matters |
|---|---|---|
| `random_uniform` | Samples uniformly from the selected label IDs. | Tests performance expected from chance under equal class probability. |
| `random_train_distribution` | Samples labels according to the training label distribution. | Reflects class imbalance without learning from text. |
| `majority_baseline` | Always predicts the most frequent training label. | Provides a strong imbalance-aware dummy baseline for accuracy comparison. |

Evaluation metrics saved for each baseline:

- accuracy
- macro-F1
- weighted-F1
- per-class precision/recall/F1 via classification report
- confusion matrix

Macro-F1 is the primary governance metric because it penalises poor performance on minority classes more clearly than accuracy.


In [ ]:
baseline_results, baseline_prediction_tables = run_baseline_experiments(

    train_df,

    test_df,

    id2label,

    paths.results_dir,

    dataset_name=DATASET_NAME,

    seed=SEED,
)


completed_results.extend(baseline_results)

prediction_tables.update(baseline_prediction_tables)

if baseline_results:
    display(pd.DataFrame(baseline_results)[["model_name", "accuracy", "macro_f1", "weighted_f1", "notes"]])

## 6. Classical TF-IDF Models

This stage trains sparse-text supervised models using TF-IDF features. These models are fast, interpretable at the feature level, and provide strong non-neural baselines for legal text classification.


Selection protocol:

1. Train each configuration on the LEDGAR training split.
2. Select the best configuration using validation macro-F1.
3. Evaluate selected models on the test split once.
4. Save the best classical pipeline and vectorizer artifacts separately.

Explainability note: TF-IDF models are useful for coursework governance because their decisions are linked to sparse lexical features rather than hidden contextual embeddings.


Variable Initialization

In [ ]:
# Initialize variables to track the best classical model and its name, which will be updated after running classical experiments.

best_classical_model = None

best_classical_name = None

Training for Classical Machine-Learning Models

Feature extraction hyperparameters:

| Hyperparameter | Values |
|---|---|
| `tokenizer` | `negation_aware` lab-grounded tokenizer |
| `max_features` | `10000`, `30000` |
| `ngram_range` | unigram `(1, 1)`, unigram+bigram `(1, 2)` |
| `lowercase` | handled inside the tokenizer |
| `stop_words` | `None` because legal stopword-like terms can change clause meaning |

The classifier is trained only after these features are built.


### Cell 1 - Feature extraction is not modelling

The cells below inspect Bag-of-Words and TF-IDF features before any classifier is trained. This keeps the Week 3 lab feature work separate from Logistic Regression, Linear SVM, and Naive Bayes.


In [ ]:
# Build Bag-of-Words features with CountVectorizer.
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

feature_sample_texts = train_df["text"].astype(str).head(200).tolist() if not train_df.empty else [
    "Borrower shall not be liable for indirect damages.",
    "Either party may terminate this Agreement on notice.",
]

bow_vectorizer = CountVectorizer(
    tokenizer=legal_safe_tokenise,
    token_pattern=None,
    lowercase=False,
)
bow_features = bow_vectorizer.fit_transform(feature_sample_texts)

print(f"BoW matrix shape: {bow_features.shape}")
print(f"BoW vocabulary size: {len(bow_vectorizer.vocabulary_)}")


In [ ]:
# Cell 3 - Inspect non-zero BoW features for one clause.
feature_names = bow_vectorizer.get_feature_names_out()
first_bow = bow_features[0].tocoo()
first_bow_df = pd.DataFrame({
    "feature": feature_names[first_bow.col],
    "count": first_bow.data,
}).sort_values(["count", "feature"], ascending=[False, True])

print(feature_sample_texts[0])
display(first_bow_df.head(30))


In [ ]:
# Build TF-IDF unigram features.
tfidf_unigram_vectorizer = TfidfVectorizer(
    tokenizer=negation_aware_tokenise,
    token_pattern=None,
    lowercase=False,
    ngram_range=(1, 1),
    max_features=MAX_FEATURES_LIST[0],
)
tfidf_unigram_features = tfidf_unigram_vectorizer.fit_transform(feature_sample_texts)

print(f"TF-IDF unigram matrix shape: {tfidf_unigram_features.shape}")
print(tfidf_unigram_vectorizer.get_feature_names_out()[:40])


In [ ]:
# Build TF-IDF unigram+bigram features.
tfidf_unibigram_vectorizer = TfidfVectorizer(
    tokenizer=negation_aware_tokenise,
    token_pattern=None,
    lowercase=False,
    ngram_range=(1, 2),
    max_features=MAX_FEATURES_LIST[0],
)
tfidf_unibigram_features = tfidf_unibigram_vectorizer.fit_transform(feature_sample_texts)

print(f"TF-IDF unigram+bigram matrix shape: {tfidf_unibigram_features.shape}")
print(tfidf_unibigram_vectorizer.get_feature_names_out()[:40])


In [ ]:
# Compare feature matrix shapes before modelling.
feature_shape_summary = pd.DataFrame([
    {"representation": "BoW unigrams", "rows": bow_features.shape[0], "features": bow_features.shape[1]},
    {"representation": "TF-IDF unigrams", "rows": tfidf_unigram_features.shape[0], "features": tfidf_unigram_features.shape[1]},
    {"representation": "TF-IDF unigrams+bigrams", "rows": tfidf_unibigram_features.shape[0], "features": tfidf_unibigram_features.shape[1]},
])
display(feature_shape_summary)


### Cell 7 - Modelling starts below

The next cells train Logistic Regression, Linear SVM, and Naive Bayes using the TF-IDF feature setup. These classifiers are modelling steps, not preprocessing.


In [ ]:

# Training with Manual Hyperparameter Tuning

if not RUN_CLASSICAL_MODELS:
    print("RUN_CLASSICAL_MODELS doesn't exist or is set to False.")

else:

    classical_output = run_classical_experiments(

        train_df,
        # Training data for classical models

        validation_df,
        # Validation data for classical models (used for hyperparameter tuning)

        test_df,
        # Test data for classical models

        id2label,
        # Mapping from label IDs to label names

        paths.results_dir,
        # Directory to save results and artifacts

        max_features_list=MAX_FEATURES_LIST,
        # List of max_features values to try for CountVectorizer/TfidfVectorizer

        ngram_ranges=NGRAM_RANGES,
        # List of ngram_range tuples to try for CountVectorizer/TfidfVectorizer

        dataset_name=DATASET_NAME,
        # Name of the dataset (used for logging and artifact naming)

        seed=SEED,
        # Random seed for reproducibility

        run_naive_bayes=RUN_NAIVE_BAYES,
        # Whether to run Naive Bayes models (MultinomialNB, ComplementNB)

        tokenizer_name="negation_aware",
        # Lab-grounded tokenizer used inside TF-IDF feature extraction
    )


# Aggregration of classical model results and prediction tables into the overall results and prediction tables.

    completed_results.extend(classical_output["results"])
    prediction_tables.update(classical_output["prediction_tables"])

# Best Model Selection and Display

    best_classical_model = classical_output["best_model"]
    best_classical_name = classical_output["best_model_name"]



    if best_classical_model is not None:

        trained_models["best_classical"] = best_classical_model

# Print classical machine-Learning Models for
# "Model Name",
# "Accuracy",
# "Macro F1",
# "Weighted F1",
# "Notes" columns

# Only print if there are results to show, otherwise skip to avoid empty table display.

    if classical_output["results"]:

        display(pd.DataFrame(classical_output["results"])[["model_name", "accuracy", "macro_f1", "weighted_f1", "notes"]])

## 7. Neural Sequence Baseline

This optional stage trains a compact BiLSTM classifier on the processed LEDGAR splits. It uses the training split for vocabulary/model fitting, the validation split for best-checkpoint selection by macro-F1, and the test split only after selection.

The stage is disabled by default with `RUN_SEQUENCE_MODEL=False` because it is slower than the classical models and should normally be run in Colab/A100 or with the standalone script.


In [ ]:
sequence_output = {"result": None, "predictions": pd.DataFrame(), "history": pd.DataFrame(), "skip_result": None}

if RUN_SEQUENCE_MODEL:
    wandb_run = maybe_start_wandb_run(
        stage_name="04_bilstm",
        config={
            "stage": "bilstm",
            "dataset": DATASET_NAME,
            "seed": SEED,
        },
    )

    sequence_output = train_sequence_classifier(
        train_df,
        validation_df,
        test_df,
        id2label,
        paths.results_dir,
        dataset_name=DATASET_NAME,
        config=SequenceModelConfig(seed=SEED),
        run_sequence_model=True,
    )

    if sequence_output["result"] is not None:
        completed_results.append(sequence_output["result"])
        prediction_tables["bilstm"] = sequence_output["predictions"]
        display(pd.DataFrame([sequence_output["result"]])[["model_name", "accuracy", "macro_f1", "weighted_f1"]])

    finish_wandb_run()
else:
    print("BiLSTM sequence baseline is configured but not run. Set RUN_SEQUENCE_MODEL=True or run scripts/run_sequence_model.py.")


## 8. Fine-Tuned Transformer Classifier

This stage optionally fine-tunes a Hugging Face sequence-classification transformer on LEDGAR. The default model is `distilbert-base-uncased` because it is smaller and more practical for coursework hardware than full BERT-size alternatives.

Training settings used by the module:

| Setting | Value |
|---|---:|
| Model | `distilbert-base-uncased` |
| Maximum sequence length | `256` tokens |
| Learning rate | `2e-5` |
| Epochs | `3` |
| Weight decay | `0.01` |
| Batch size | `16` on larger GPUs, otherwise `8` |
| Mixed precision | `fp16=True` when CUDA is available |
| Model selection | best validation `macro_f1` |
| Early stopping | patience `1` when the callback is available |

Runtime governance:

- This section skips gracefully if CUDA/GPU is unavailable.
- Memory or environment failures are caught and recorded as skipped results.
- The transformer is evaluated on the same LEDGAR test labels as the classical models.

Explainability limitation: transformer representations are contextual but less directly inspectable than TF-IDF features, so confusion matrices and misclassified examples are important for interpreting behavior.


In [ ]:
wandb_run = maybe_start_wandb_run(
    stage_name="05_transformers_hpt_all_variants",
    config={
        "stage": "transformers_hpt",
        "dataset": DATASET_NAME,
        "seed": SEED,
        "transformer_model_name": TRANSFORMER_MODEL_NAME,
        "run_transformer_hpt": RUN_TRANSFORMER_HPT,
        "random_trials": HPT_RANDOM_TRIALS,
        "bayes_trials": HPT_BAYES_TRIALS,
    },
)

hpt_output = None

if RUN_TRANSFORMER_HPT:
    hpt_output = run_two_stage_transformer_hpt(
        train_df,
        validation_df,
        test_df,
        id2label,
        paths.results_dir,
        dataset_name=DATASET_NAME,
        config=TransformerHPTConfig(
            model_name=TRANSFORMER_MODEL_NAME,
            random_trials=HPT_RANDOM_TRIALS,
            bayes_trials=HPT_BAYES_TRIALS,
            seed=SEED,
            early_stopping_patience=1,
            save_total_limit=1,
            final_retrain=True,
        ),
        wandb_enabled=WANDB_ACTIVE,
        wandb_project=WANDB_PROJECT,
        wandb_entity=WANDB_ENTITY,
        wandb_mode=WANDB_MODE,
    )

    transformer_output = hpt_output.get("final_output") or {
        "result": None,
        "predictions": pd.DataFrame(),
        "trainer": None,
        "skip_result": {
            "model_family": "transformer",
            "model_name": TRANSFORMER_MODEL_NAME,
            "training_type": "fine-tuned supervised",
            "dataset": DATASET_NAME,
            "eval_split": "test",
            "sample_size": 0,
            "accuracy": np.nan,
            "macro_f1": np.nan,
            "weighted_f1": np.nan,
            "notes": f"Transformer HPT did not produce a final model: {hpt_output.get('reason', 'unknown')}",
        },
    }
else:
    transformer_output = train_transformer_classifier(
        train_df,
        validation_df,
        test_df,
        id2label,
        paths.results_dir,
        model_name=TRANSFORMER_MODEL_NAME,
        max_length=MAX_TRANSFORMER_LENGTH,
        dataset_name=DATASET_NAME,
        seed=SEED,
        run_transformer=RUN_TRANSFORMER,
        wandb_enabled=WANDB_ACTIVE,
        wandb_run_name=getattr(wandb_run, "name", None),
    )

finish_wandb_run()


In [ ]:

# If transformer_output contains a valid output,
#
#   append the results to completed_results,
#   update the prediction_tables with the transformer's predictions,
#   store the trained transformer model in trained_models under the key "transformer_trainer",
#   and display a DataFrame with the transformer's results showing:
#
#       "model_name",
#       "accuracy",
#       "macro_f1",
#       "weighted_f1" columns.

if transformer_output["result"] is not None:
    completed_results.append(transformer_output["result"])
    prediction_tables[TRANSFORMER_MODEL_NAME] = transformer_output["predictions"]
    trained_models["transformer_trainer"] = transformer_output["trainer"]
    display(pd.DataFrame([transformer_output["result"]])[["model_name",
                                                          "accuracy",
                                                          "macro_f1",
                                                          "weighted_f1"]])


elif transformer_output["skip_result"] is not None:
    completed_results.append(transformer_output["skip_result"])

## 9. Qwen2.5-Instruct Prompting Baseline

This stage optionally evaluates an instruction-tuned language model as a prompting baseline. Qwen is not fine-tuned; it is only prompted to classify clauses into the fixed LEDGAR label set.

Prompting setup:

| Mode | Description |
|---|---|
| Zero-shot | Provides the clause text and full list of allowed labels. |
| Static few-shot | Adds training examples only; validation and test examples are never used as demonstrations. |
| Retrieval few-shot | Retrieves similar examples from the training split only. |

Generation and parsing controls:

| Setting | Value |
|---|---:|
| Model | `Qwen/Qwen2.5-3B-Instruct` |
| Evaluation sample | up to `200` test examples, sampled with `SEED=42` |
| Decoding | deterministic, `do_sample=False` |
| New tokens | `max_new_tokens=20` |
| Output requirement | return exactly one allowed label |
| Parser | exact allowed-label match after whitespace/case normalisation |
| Invalid outputs | marked as `INVALID_PREDICTION` and reported separately |

Governance note: this is not directly equivalent to supervised fine-tuning. The prompted model has different pretraining and task setup, so results should be interpreted as a separate baseline rather than a perfectly fair model-family comparison.


In [ ]:
wandb_run = maybe_start_wandb_run(
    stage_name="06_qwen_prompting",
    config={
        "stage": "qwen_prompting",
        "dataset": DATASET_NAME,
        "seed": SEED,
        "qwen_model_name": QWEN_MODEL_NAME,
        "qwen_eval_sample_size": QWEN_EVAL_SAMPLE_SIZE,
    },
)



qwen_output = run_qwen_baseline(

    train_df,
    # Training data for Qwen prompting (used to create few-shot examples)

    test_df,
    # Test data for Qwen prompting (used for evaluation)

    label2id,
    # Mapping from label names to label IDs, which may be needed for formatting prompts or interpreting outputs

    id2label,
    # Mapping from label IDs to label names, which may be needed for formatting prompts or interpreting outputs

    paths.results_dir,
    # Directory to save results and artifacts related to the Qwen baseline

    model_name=QWEN_MODEL_NAME,
    # Name of the Qwen model to use for prompting (e.g., "Qwen/Qwen2.5-3B-Instruct")

    label_names=label_names,
    # List of label names corresponding to the label IDs, which may be needed for formatting prompts or interpreting outputs

    eval_sample_size=QWEN_EVAL_SAMPLE_SIZE,
    # Number of test samples to evaluate on for the Qwen baseline (e.g., 200)

    few_shot_examples_per_class=QWEN_FEW_SHOT_EXAMPLES_PER_CLASS,
    # Number of few-shot examples to include per class in the prompt for Qwen (e.g., 1)

    dataset_name=DATASET_NAME,
    # Name of the dataset (used for logging and artifact naming)

    seed=SEED,
    # Random seed for reproducibility, which may be used for sampling evaluation examples or shuffling data

    run_qwen=RUN_QWEN_BASELINE,
    # Whether to run the Qwen baseline (if False, the function may skip execution and return None or a skip result)
)



In [ ]:

completed_results.extend(qwen_output["results"])

qwen_predictions_df = qwen_output["predictions"]

qwen_invalid_outputs_df = qwen_output["invalid_outputs"]

qwen_model = qwen_output["model"]

qwen_tokenizer = qwen_output["tokenizer"]

if qwen_output["results"]:
    display(pd.DataFrame(qwen_output["results"])[[
        "model_name",
        "accuracy",
        "macro_f1",
        "weighted_f1",
        "notes"]])

#
# If qwen_output contains valid results,
#   append the results to completed_results,
#   store the predictions, invalid outputs, model, and tokenizer in respective variables,
#   and display a DataFrame with the Qwen baseline results showing:
#       "model_name",
#       "accuracy",
#       "macro_f1",
#       "weighted_f1",
#       "notes" columns.
#

finish_wandb_run()


## 10. Small Agentic Review Prototype

This stage demonstrates a small human-in-the-loop clause triage workflow. It is inspired by tool-use/ReAct-style ideas, but it is not a large autonomous agent and it does not provide legal advice.

Workflow:

1. Use the best available supervised classifier to predict a clause type.
2. Estimate prediction confidence where the model supports it.
3. Flag examples below the review threshold as requiring human review.
4. Optionally ask Qwen for a short triage explanation when Qwen loaded successfully.

Prototype settings:

| Setting | Value |
|---|---:|
| Sample size | `20` test examples |
| Review threshold | `0.55` confidence |
| Logistic Regression confidence | maximum predicted probability |
| Linear SVM confidence | transformed decision-function margin |
| Required disclaimer | `This output is for clause triage and research purposes only.` |

Governance limitation: this section is illustrative. It should not be treated as a production legal review system, risk model, or legal advisor.


In [ ]:
agentic_examples_df = run_agentic_review(

    test_df,
    # Test data for agentic review (used to identify examples for review and potential correction)

    id2label,
    # Mapping from label IDs to label names, which may be needed for formatting prompts or interpreting outputs

    paths.results_dir,

    # Directory to save results and artifacts related to the agentic review
    best_model=best_classical_model,

    # The best classical model identified from the classical experiments, which may be used as a baseline for comparison or to identify examples where it fails
    qwen_model=qwen_model,

    # The Qwen model used for prompting, which may be used to generate explanations or corrections for misclassified examples
    qwen_tokenizer=qwen_tokenizer,

    # Name of the dataset (used for logging and artifact naming)
    run_agentic=RUN_AGENTIC_EXTENSION,

    # Random seed for reproducibility, which may be used for sampling examples for review or shuffling data
    seed=SEED,

    # Whether to run the agentic review process (if False, the function may skip execution and return an empty DataFrame or None
)

if not agentic_examples_df.empty:
    display(agentic_examples_df.head(10))

## 11. Final Model Comparison

This stage consolidates all completed and skipped model runs into one comparison table. It does not insert or fabricate any metrics; it only formats rows produced by earlier sections.

Comparison columns:

| Column | Meaning |
|---|---|
| `model_family` | baseline, classical, transformer, or prompting family. |
| `model_name` | specific model/configuration name. |
| `training_type` | dummy, supervised, fine-tuned, prompted, or skipped. |
| `dataset` | evaluation dataset, here LEDGAR for the main experiment. |
| `eval_split` | split used for reported metrics, usually test. |
| `sample_size` | number of evaluated examples. |
| `accuracy` | overall exact-label accuracy. |
| `macro_f1` | unweighted mean F1 across classes; primary metric. |
| `weighted_f1` | class-frequency-weighted F1. |
| `notes` | skip reason or relevant run detail. |

The table and macro-F1 plot are saved under `results/` for later inspection.


In [ ]:
comparison_df = save_final_comparison(completed_results, paths.results_dir)

print(f"Saved final comparison to: {paths.results_dir / 'final_model_comparison.csv'}")

display(comparison_df)

## 12. Error Analysis

This stage checks model behaviour beyond aggregate metrics. It uses saved predictions, confusion outputs, and class counts from earlier stages.

Outputs reviewed:

| Output | Purpose |
|---|---|
| Top confused label pairs | Shows where the best classical model mixes labels. |
| Misclassified examples | Provides examples for qualitative inspection. |
| Transformer errors | Included only when the transformer ran. |
| Qwen invalid outputs | Included only when Qwen produced real predictions. |
| Class imbalance summary | Shows how training labels are distributed. |

These outputs support later discussion without writing report conclusions here.


In [ ]:
error_outputs = run_error_analysis(
    comparison_df,
    prediction_tables,
    train_df,
    paths.results_dir,
    best_classical_name=best_classical_name,
    transformer_model_name=TRANSFORMER_MODEL_NAME,
    qwen_predictions_df=qwen_predictions_df,
    qwen_invalid_outputs_df=qwen_invalid_outputs_df,
)


Interpretation focus:

| Issue | Why it matters |
|---|---|
| Class imbalance | Accuracy can look high while minority classes perform poorly. |
| Label ambiguity | Legal clauses may plausibly fit more than one clause type. |
| Long clauses | Transformer truncation and TF-IDF sparsity can affect predictions. |
| Boilerplate wording | Repeated legal phrasing can make labels harder to separate. |
| Invalid LLM outputs | Prompted models may ignore the closed label set. |

In [ ]:
print(f"Best completed model by macro-F1: {error_outputs.get('best_model_name')}")

In [ ]:
if "classical_confusions" in error_outputs:
    print("Top classical confused label pairs:")
    display(error_outputs["classical_confusions"])

if "classical_misclassified" in error_outputs:
    print("Classical misclassified examples:")
    display(error_outputs["classical_misclassified"][["text", "label", "predicted_label"]])

if "transformer_misclassified" in error_outputs:
    print("Transformer misclassified examples:")
    display(error_outputs["transformer_misclassified"][["text", "label", "predicted_label"]])

if "qwen_invalid_outputs" in error_outputs:
    print("Qwen invalid outputs:")
    display(error_outputs["qwen_invalid_outputs"].head(10))

if "qwen_plausible_nonmatching" in error_outputs:
    print("Qwen plausible non-matching label examples for manual inspection:")
    display(error_outputs["qwen_plausible_nonmatching"].head(10))

print("Class imbalance summary:")
display(error_outputs.get("class_imbalance", pd.DataFrame()).head(20))


## 13. Report Artifact Exports

This stage writes the report-facing tables, figures, prompt outputs, and environment metadata. It does not edit `report.tex` or create metrics that were not produced earlier.

Key exports:

| Export type | Destination |
|---|---|
| CSV report tables | `outputs/` |
| Figures | `figures/` and `outputs/figures/` |
| Predictions and prompts | `outputs/` and `outputs/predictions/` |
| Runtime metadata | `outputs/environment.json` |

Skipped transformer or Qwen runs remain marked as skipped in the exported evidence.


In [ ]:
from modules.report_exports import export_report_artifacts

In [ ]:
report_artifacts = export_report_artifacts(
    paths=paths,
    processed_splits=processed_splits,
    label2id=label2id,
    id2label=id2label,
    completed_results=completed_results,
    prediction_tables=prediction_tables,
    error_outputs=error_outputs,
    qwen_predictions_df=qwen_predictions_df,
    qwen_invalid_outputs_df=qwen_invalid_outputs_df,
    seed=SEED,
    dataset_name=DATASET_NAME,
    max_features_list=MAX_FEATURES_LIST,
    ngram_ranges=NGRAM_RANGES,
    transformer_model_name=TRANSFORMER_MODEL_NAME,
    max_transformer_length=MAX_TRANSFORMER_LENGTH,
    qwen_model_name=QWEN_MODEL_NAME,
    qwen_eval_sample_size=QWEN_EVAL_SAMPLE_SIZE,
    qwen_few_shot_examples_per_class=QWEN_FEW_SHOT_EXAMPLES_PER_CLASS,
    run_naive_bayes=RUN_NAIVE_BAYES,
)

print("Saved report artifacts:")
for artifact_name, artifact_path in report_artifacts.items():
    print(f"- {artifact_name}: {artifact_path}")

# W&B is intentionally not used in the report-export stage.
# This avoids false W&B crashes and oversized artifact logging in Colab.
print("W&B report-export logging skipped; local report artifacts were saved above.")
finish_wandb_run()


## 14. Method Notes and Limitations

These notes record modelling choices for transparency. They are not final coursework conclusions.

| Area | Note |
|---|---|
| Aim | Compare dummy baselines, classical supervised models, optional transformer fine-tuning, and optional Qwen prompting for LEDGAR clause classification. |
| Dataset | LEDGAR is the main supervised dataset. CUAD is separate and not merged into LEDGAR. |
| Preprocessing | The pipeline normalises whitespace, removes empty rows and exact duplicate text-label pairs, selects top-k training labels, and preserves official splits. |
| Baselines | Random and majority models provide lower-bound references. |
| Classical models | TF-IDF Logistic Regression, Linear SVM, and optional Naive Bayes provide sparse-text baselines. |
| Transformer | DistilBERT or LegalBERT is fine-tuned only when the runtime supports it. |
| Qwen | Qwen2.5-Instruct is used as zero-shot/few-shot prompting, not training. |
| Agentic extension | The prototype flags low-confidence examples for human review and is not legal advice. |

Limitations to discuss after results are generated: class imbalance, label ambiguity, long-clause truncation, prompt sensitivity, and differences between supervised and prompted model setups.


In [ ]:
# Locate project root.
try:
    PROJECT_ROOT_CHECK = Path(paths.project_root)
except NameError:
    PROJECT_ROOT_CHECK = Path.cwd()

REPORT_TEX = Path(r"C:\Users\ybenj\Downloads\report.tex")

# In Colab, this path only exists if report.tex was uploaded or synced.
# Standard report-facing artifacts are checked either way.
print(f"Project root: {PROJECT_ROOT_CHECK}")
print(f"report.tex found: {REPORT_TEX.exists()} -> {REPORT_TEX}")

# Required report-facing tables and files.
required_files = [
    "outputs/data_summary.csv",
    "outputs/label_distribution.csv",
    "outputs/main_results.csv",
    "outputs/per_class_results.csv",
    "outputs/confusion_pairs.csv",
    "outputs/misclassified_examples.csv",
    "outputs/hyperparameters.csv",
    "outputs/environment.json",
    "outputs/report_artifact_manifest.json",
    "data/processed/dataset_summary.json",
    "data/processed/label_counts.json",
    "data/processed/label_names.txt",
    "data/processed/ledgar_train.jsonl",
    "data/processed/ledgar_validation.jsonl",
    "data/processed/ledgar_test.jsonl",
]

# Conditional model evidence may contain skipped-status rows.
optional_but_expected_files = [
    "outputs/transformer_results.csv",
    "outputs/transformer_predictions.csv",
    "outputs/qwen_results.csv",
    "outputs/qwen_predictions.csv",
    "outputs/qwen_invalid_outputs.csv",
    "outputs/qwen_prompt_examples.txt",
    "results/transformer/runtime.json",
    "results/transformer/training_args.json",
    "results/transformer/training_log_history.json",
    "results/qwen/runtime.json",
    "results/qwen/qwen_run_config.json",
]

# Figures expected by the current report.
standard_report_figures = [
    "figures/label_distribution.png",
    "figures/clause_length_distribution.png",
    "figures/agentic_review_workflow.png",
    "figures/model_comparison_macro_f1.png",
    "figures/confusion_matrix_best_model.png",
    "figures/qwen_invalid_predictions.png",
]

# Add figure refs from report.tex when available.
figure_refs_from_tex = []
if REPORT_TEX.exists():
    tex = REPORT_TEX.read_text(encoding="utf-8", errors="ignore")
    figure_refs_from_tex = re.findall(r"\\includegraphics(?:\[[^\]]*\])?\{([^}]+)\}", tex)

figure_refs = sorted(set(standard_report_figures + figure_refs_from_tex))

def file_status(relative_path):
    path = PROJECT_ROOT_CHECK / relative_path
    exists = path.exists()
    size = path.stat().st_size if exists and path.is_file() else 0
    return {
        "path": relative_path,
        "exists": exists,
        "non_empty": bool(exists and size > 0),
        "size_bytes": size,
    }

def csv_rows(relative_path):
    path = PROJECT_ROOT_CHECK / relative_path
    if not path.exists() or path.stat().st_size == 0:
        return None
    try:
        return len(pd.read_csv(path))
    except Exception:
        return "unreadable"

def jsonl_rows(relative_path):
    path = PROJECT_ROOT_CHECK / relative_path
    if not path.exists() or path.stat().st_size == 0:
        return None
    return sum(1 for line in path.open("r", encoding="utf-8") if line.strip())

# Check files.
required_status = pd.DataFrame([file_status(p) for p in required_files])
optional_status = pd.DataFrame([file_status(p) for p in optional_but_expected_files])

required_status["rows_if_csv"] = required_status["path"].apply(lambda p: csv_rows(p) if p.endswith(".csv") else None)
optional_status["rows_if_csv"] = optional_status["path"].apply(lambda p: csv_rows(p) if p.endswith(".csv") else None)

print("\nREQUIRED FILES")
display(required_status)

print("\nOPTIONAL / CONDITIONAL MODEL EVIDENCE FILES")
display(optional_status)

# Check figures in report and output locations.
figure_rows = []
for ref in figure_refs:
    ref_path = Path(ref)
    candidates = [
        PROJECT_ROOT_CHECK / ref,
        PROJECT_ROOT_CHECK / "outputs" / ref,
    ]
    # Also check outputs/figures/foo.png for figures/foo.png refs.
    if len(ref_path.parts) >= 2 and ref_path.parts[0] == "figures":
        candidates.append(PROJECT_ROOT_CHECK / "outputs" / "figures" / ref_path.name)

    existing = [p for p in candidates if p.exists() and p.is_file() and p.stat().st_size > 0]
    figure_rows.append({
        "figure_ref": ref,
        "found": bool(existing),
        "found_at": str(existing[0]) if existing else "",
        "size_bytes": existing[0].stat().st_size if existing else 0,
    })

figure_status = pd.DataFrame(figure_rows)
print("\nFIGURES")
display(figure_status)

# Check prediction files against processed test size.
test_rows = jsonl_rows("data/processed/ledgar_test.jsonl")
prediction_dir = PROJECT_ROOT_CHECK / "outputs" / "predictions"
prediction_rows = []

if prediction_dir.exists():
    for pred_path in sorted(prediction_dir.glob("*_test_predictions.jsonl")):
        count = sum(1 for line in pred_path.open("r", encoding="utf-8") if line.strip())
        prediction_rows.append({
            "prediction_file": str(pred_path.relative_to(PROJECT_ROOT_CHECK)),
            "rows": count,
            "matches_test_rows": count == test_rows,
            "expected_test_rows": test_rows,
        })

prediction_status = pd.DataFrame(prediction_rows)
print("\nPREDICTION ROW COUNTS")
display(prediction_status)

# Final pass/fail summary.
missing_required = required_status[~required_status["non_empty"]]
missing_figures = figure_status[~figure_status["found"]]
bad_predictions = prediction_status[prediction_status["matches_test_rows"] == False] if not prediction_status.empty else pd.DataFrame()

print("\nSUMMARY")
print(f"Required files OK: {missing_required.empty}")
print(f"Figures OK: {missing_figures.empty}")
print(f"Prediction row counts OK: {bad_predictions.empty}")
print(f"Processed test rows: {test_rows}")

if not missing_required.empty:
    print("\nMissing/empty required files:")
    display(missing_required)

if not missing_figures.empty:
    print("\nMissing figures:")
    display(missing_figures)

if not bad_predictions.empty:
    print("\nPrediction files with wrong row counts:")
    display(bad_predictions)


In [ ]:
from pathlib import Path

# @title

# Final report evidence audit cell
# Run before using the notebook outputs to fill report.tex.


# In Colab, set this manually if report.tex is uploaded or synced.
REPORT_TEX_PATH = "/content/drive/MyDrive/.../report.tex"

try:
    PROJECT_ROOT_CHECK = Path(paths.project_root)
except NameError:
    PROJECT_ROOT_CHECK = Path.cwd()

REPORT_TEX = Path(REPORT_TEX_PATH)

print(f"Project root: {PROJECT_ROOT_CHECK}")
print(f"report.tex found: {REPORT_TEX.exists()} -> {REPORT_TEX}")

def safe_display(title, df):
    print(f"\n{title}")
    try:
        display(df)
    except NameError:
        print(df.to_string(index=False))


def file_mtime(path):
    if not path.exists():
        return None
    return datetime.fromtimestamp(path.stat().st_mtime, tz=timezone.utc)


def count_jsonl(path):
    if not path.exists() or path.stat().st_size == 0:
        return None
    return sum(1 for line in path.open("r", encoding="utf-8") if line.strip())


def read_csv_safe(path):
    if not path.exists() or path.stat().st_size == 0:
        return None
    try:
        return pd.read_csv(path)
    except Exception as exc:
        print(f"Could not read CSV {path}: {type(exc).__name__}: {exc}")
        return None


def read_json_safe(path):
    if not path.exists() or path.stat().st_size == 0:
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"Could not read JSON {path}: {type(exc).__name__}: {exc}")
        return None


def resolve_report_path(value):
    if pd.isna(value) or not str(value).strip():
        return None

    raw = str(value)
    direct = Path(raw)
    if direct.exists():
        return direct

    # Handle paths saved in Colab like /content/drive/.../results/...
    for marker in ["outputs/", "results/", "figures/", "data/processed/"]:
        if marker in raw.replace("\\", "/"):
            rel = raw.replace("\\", "/").split(marker, 1)[1]
            candidate = PROJECT_ROOT_CHECK / marker.rstrip("/") / rel
            if candidate.exists():
                return candidate

    candidate = PROJECT_ROOT_CHECK / raw
    if candidate.exists():
        return candidate

    return None


def safe_name(value):
    return re.sub(r"[^a-zA-Z0-9]+", "_", str(value).lower()).strip("_") or "item"


# 1. Required artifact checks

required_specs = {
    "outputs/data_summary.csv": ["metric", "value", "notes"],
    "outputs/label_distribution.csv": ["label", "label_id", "train_count", "validation_count", "test_count", "total_count"],
    "outputs/main_results.csv": ["model_family", "model_name", "sample_size", "accuracy", "macro_f1", "weighted_f1"],
    "outputs/per_class_results.csv": ["model_family", "model_name", "label", "precision", "recall", "f1_score", "support"],
    "outputs/confusion_pairs.csv": ["rank", "true_label", "predicted_label", "count"],
    "outputs/misclassified_examples.csv": ["text", "label", "predicted_label"],
    "outputs/hyperparameters.csv": ["component", "parameter", "value"],
    "outputs/environment.json": None,
    "outputs/leakage_audit.json": None,
    "outputs/report_artifact_manifest.json": None,
    "data/processed/dataset_summary.json": None,
    "data/processed/label_counts.json": None,
    "data/processed/label_names.txt": None,
    "data/processed/ledgar_train.jsonl": None,
    "data/processed/ledgar_validation.jsonl": None,
    "data/processed/ledgar_test.jsonl": None,
    "results/final_model_comparison.csv": ["model_family", "model_name", "sample_size", "accuracy", "macro_f1", "weighted_f1"],
}

artifact_rows = []
for rel_path, expected_cols in required_specs.items():
    path = PROJECT_ROOT_CHECK / rel_path
    exists = path.exists()
    non_empty = exists and path.is_file() and path.stat().st_size > 0
    columns_ok = True
    rows = None
    missing_cols = []

    if non_empty and rel_path.endswith(".csv"):
        df = read_csv_safe(path)
        if df is not None:
            rows = len(df)
            missing_cols = [c for c in expected_cols if c not in df.columns]
            columns_ok = not missing_cols
        else:
            columns_ok = False

    if non_empty and rel_path.endswith(".jsonl"):
        rows = count_jsonl(path)

    artifact_rows.append({
        "path": rel_path,
        "exists": exists,
        "non_empty": non_empty,
        "rows": rows,
        "columns_ok": columns_ok,
        "missing_columns": ", ".join(missing_cols),
        "modified_utc": file_mtime(path),
    })

artifact_status = pd.DataFrame(artifact_rows)
safe_display("1. REQUIRED ARTIFACTS", artifact_status)


# 2. Split consistency checks

split_paths = {
    "train": PROJECT_ROOT_CHECK / "data/processed/ledgar_train.jsonl",
    "validation": PROJECT_ROOT_CHECK / "data/processed/ledgar_validation.jsonl",
    "test": PROJECT_ROOT_CHECK / "data/processed/ledgar_test.jsonl",
}
split_counts = {split: count_jsonl(path) for split, path in split_paths.items()}

data_summary = read_csv_safe(PROJECT_ROOT_CHECK / "outputs/data_summary.csv")
summary_counts = {}

if data_summary is not None:
    summary_map = dict(zip(data_summary["metric"], data_summary["value"]))
    for split in ["train", "validation", "test"]:
        key = f"filtered_{split}_examples"
        try:
            summary_counts[split] = int(float(summary_map.get(key)))
        except Exception:
            summary_counts[split] = None

split_status = pd.DataFrame([
    {
        "split": split,
        "processed_jsonl_rows": split_counts.get(split),
        "data_summary_rows": summary_counts.get(split),
        "matches_data_summary": split_counts.get(split) == summary_counts.get(split),
    }
    for split in ["train", "validation", "test"]
])
safe_display("2. SPLIT CONSISTENCY", split_status)


# ----------------------------
# 3. Leakage audit check
# ----------------------------

leakage = read_json_safe(PROJECT_ROOT_CHECK / "outputs/leakage_audit.json")
leakage_rows = []

if leakage:
    after = leakage.get("cross_split_overlaps_after_deduplication", {})
    for pair, values in after.items():
        leakage_rows.append({
            "pair": pair,
            "text_overlap": values.get("text_overlap"),
            "text_label_overlap": values.get("text_label_overlap"),
            "ok_zero_overlap": values.get("text_overlap") == 0 and values.get("text_label_overlap") == 0,
        })

leakage_status = pd.DataFrame(leakage_rows)
safe_display("3. LEAKAGE AUDIT", leakage_status)


# ----------------------------
# 4. Metrics consistency checks
# ----------------------------

main_results = read_csv_safe(PROJECT_ROOT_CHECK / "outputs/main_results.csv")
final_comparison = read_csv_safe(PROJECT_ROOT_CHECK / "results/final_model_comparison.csv")

metric_consistency_rows = []

if main_results is not None and final_comparison is not None:
    keys = ["model_family", "model_name"]
    metric_cols = ["sample_size", "accuracy", "macro_f1", "weighted_f1"]
    merged = main_results[keys + metric_cols].merge(
        final_comparison[keys + metric_cols],
        on=keys,
        suffixes=("_outputs", "_results"),
        how="outer",
        indicator=True,
    )

    for _, row in merged.iterrows():
        ok = row["_merge"] == "both"
        diffs = []
        if ok:
            for col in metric_cols:
                left = row[f"{col}_outputs"]
                right = row[f"{col}_results"]
                if pd.isna(left) and pd.isna(right):
                    continue
                if col == "sample_size":
                    same = int(left) == int(right)
                else:
                    same = abs(float(left) - float(right)) < 1e-9
                if not same:
                    ok = False
                    diffs.append(col)

        metric_consistency_rows.append({
            "model_family": row.get("model_family"),
            "model_name": row.get("model_name"),
            "present_in_both": row["_merge"] == "both",
            "metrics_match": ok,
            "different_columns": ", ".join(diffs),
        })

metric_consistency = pd.DataFrame(metric_consistency_rows)
safe_display("4. METRIC CONSISTENCY: outputs/main_results.csv vs results/final_model_comparison.csv", metric_consistency)


# ----------------------------
# 5. Classification report/confusion matrix existence
# ----------------------------

report_rows = []

if main_results is not None:
    for _, row in main_results.iterrows():
        sample_size = row.get("sample_size")
        macro_f1 = row.get("macro_f1")
        is_completed = pd.notna(macro_f1) and pd.notna(sample_size) and int(sample_size) > 0

        report_path = resolve_report_path(row.get("classification_report_path"))
        cm_path = resolve_report_path(row.get("confusion_matrix_path"))

        report_rows.append({
            "model_name": row.get("model_name"),
            "completed_result": is_completed,
            "classification_report_found": bool(report_path) if is_completed else "not_required_if_skipped",
            "confusion_matrix_found": bool(cm_path) if is_completed else "not_required_if_skipped",
            "classification_report_path": str(report_path) if report_path else "",
            "confusion_matrix_path": str(cm_path) if cm_path else "",
        })

report_status = pd.DataFrame(report_rows)
safe_display("5. REPORT + CONFUSION MATRIX EVIDENCE", report_status)


# ----------------------------
# 6. Prediction file row counts
# ----------------------------

test_rows = split_counts.get("test")
prediction_rows = []
prediction_dir = PROJECT_ROOT_CHECK / "outputs/predictions"

if prediction_dir.exists():
    for pred_path in sorted(prediction_dir.glob("*_test_predictions.jsonl")):
        rows = count_jsonl(pred_path)
        prediction_rows.append({
            "prediction_file": str(pred_path.relative_to(PROJECT_ROOT_CHECK)),
            "rows": rows,
            "expected_test_rows": test_rows,
            "matches_test_rows": rows == test_rows,
            "modified_utc": file_mtime(pred_path),
        })

prediction_status = pd.DataFrame(prediction_rows)
safe_display("6. PREDICTION ROW COUNTS", prediction_status)


# ----------------------------
# 7. Skipped transformer/Qwen guard
# ----------------------------

guard_rows = []

if main_results is not None:
    for model_group, mask in {
        "transformer": main_results["model_family"].astype(str).str.contains("transformer", case=False, na=False)
                       | main_results["model_name"].astype(str).str.contains("bert|distilbert|legal", case=False, na=False),
        "qwen": main_results["model_name"].astype(str).str.contains("qwen", case=False, na=False)
                | main_results["model_family"].astype(str).str.contains("qwen|llm", case=False, na=False),
    }.items():
        subset = main_results[mask]
        if subset.empty:
            guard_rows.append({
                "model_group": model_group,
                "present": False,
                "real_result_rows": 0,
                "skipped_or_empty_rows": 0,
                "safe_to_claim_results": False,
            })
        else:
            real = subset[pd.to_numeric(subset["sample_size"], errors="coerce").fillna(0).gt(0) & subset["macro_f1"].notna()]
            guard_rows.append({
                "model_group": model_group,
                "present": True,
                "real_result_rows": len(real),
                "skipped_or_empty_rows": len(subset) - len(real),
                "safe_to_claim_results": len(real) > 0,
            })

skipped_guard = pd.DataFrame(guard_rows)
safe_display("7. SKIPPED MODEL GUARD", skipped_guard)


# ----------------------------
# 8. Figure reference checks
# ----------------------------

standard_figures = [
    "figures/label_distribution.png",
    "figures/clause_length_distribution.png",
    "figures/agentic_review_workflow.png",
    "figures/model_comparison_macro_f1.png",
    "figures/confusion_matrix_best_model.png",
    "figures/qwen_invalid_predictions.png",
]

figure_refs_from_tex = []
todo_rows = []

if REPORT_TEX.exists():
    tex = REPORT_TEX.read_text(encoding="utf-8", errors="ignore")
    figure_refs_from_tex = re.findall(r"\\includegraphics(?:\\[[^\]]*\\])?\{([^}]+)\}", tex)

    for line_no, line in enumerate(tex.splitlines(), 1):
        for todo in re.findall(r"\\todo\{([^}]*)\}", line):
            todo_rows.append({
                "line": line_no,
                "todo": todo,
                "likely_evidence": (
                    "outputs/main_results.csv" if any(x in todo.lower() for x in ["result", "score", "model", "f1"]) else
                    "outputs/data_summary.csv" if any(x in todo.lower() for x in ["example", "length", "train", "validation", "test"]) else
                    "outputs/per_class_results.csv" if "class" in todo.lower() or "categor" in todo.lower() else
                    "outputs/confusion_pairs.csv" if "label" in todo.lower() else
                    "manual review needed"
                ),
            })

figure_refs = sorted(set(standard_figures + figure_refs_from_tex))

qwen_real = False
if main_results is not None:
    qwen_mask = main_results["model_name"].astype(str).str.contains("qwen", case=False, na=False)
    qwen_subset = main_results[qwen_mask]
    if not qwen_subset.empty:
        qwen_real = bool(
            pd.to_numeric(qwen_subset["sample_size"], errors="coerce").fillna(0).gt(0).any()
            and qwen_subset["macro_f1"].notna().any()
        )

figure_rows = []
for ref in figure_refs:
    ref_path = Path(ref)
    candidates = [
        PROJECT_ROOT_CHECK / ref,
        PROJECT_ROOT_CHECK / "outputs" / ref,
    ]
    if len(ref_path.parts) >= 2 and ref_path.parts[0] == "figures":
        candidates.append(PROJECT_ROOT_CHECK / "outputs" / "figures" / ref_path.name)

    existing = [p for p in candidates if p.exists() and p.is_file() and p.stat().st_size > 0]
    is_qwen_invalid = ref_path.name == "qwen_invalid_predictions.png"

    figure_rows.append({
        "figure_ref": ref,
        "found": bool(existing),
        "conditional_qwen_figure": is_qwen_invalid,
        "qwen_real_result_available": qwen_real,
        "ok_for_pipeline": bool(existing) or (is_qwen_invalid and not qwen_real),
        "report_edit_warning": "remove/keep TODO if Qwen skipped" if is_qwen_invalid and not qwen_real else "",
        "found_at": str(existing[0]) if existing else "",
        "size_bytes": existing[0].stat().st_size if existing else 0,
    })

figure_status = pd.DataFrame(figure_rows)
safe_display("8. FIGURE REFERENCES", figure_status)


# ----------------------------
# 9. TODO evidence map
# ----------------------------

todo_status = pd.DataFrame(todo_rows)
safe_display("9. REPORT TODO EVIDENCE MAP", todo_status)


# ----------------------------
# 10. Final strict summary
# ----------------------------

critical_failures = []

if not artifact_status["non_empty"].all():
    critical_failures.append("Missing or empty required artifacts.")

if not artifact_status["columns_ok"].all():
    critical_failures.append("Some required CSVs are missing expected columns.")

if not split_status["matches_data_summary"].all():
    critical_failures.append("Processed split counts do not match outputs/data_summary.csv.")

if not leakage_status.empty and not leakage_status["ok_zero_overlap"].all():
    critical_failures.append("Leakage audit still shows cross-split overlaps.")

if not metric_consistency.empty and not metric_consistency["metrics_match"].all():
    critical_failures.append("outputs/main_results.csv and results/final_model_comparison.csv disagree.")

if not prediction_status.empty and not prediction_status["matches_test_rows"].all():
    critical_failures.append("One or more test prediction files do not match processed test row count.")

completed_missing_reports = report_status[
    (report_status["completed_result"] == True)
    &
        ((report_status["classification_report_found"] != True)
        | (report_status["confusion_matrix_found"] != True))
]
if not completed_missing_reports.empty:
    critical_failures.append("A completed model is missing a classification report or confusion matrix.")

blocking_figures = figure_status[figure_status["ok_for_pipeline"] != True]
if not blocking_figures.empty:
    critical_failures.append("One or more required non-conditional figures are missing.")

print("\nFINAL SUMMARY")
print(f"Critical failures: {len(critical_failures)}")
for failure in critical_failures:
    print(f"- {failure}")

if not critical_failures:
    print("PASS: report-facing evidence looks internally consistent.")
else:
    print("FAIL: fix the issues above before asking Codex to fill report.tex.")

## Final W&B Cleanup

This cell safely closes any W&B run that may still be open after interrupted or partial execution.


In [ ]:
# Final W&B cleanup for Colab partial/staged execution.
finish_wandb_run()
